# 05 — Financial Documents and Chunking Laboratory

**First Finance - Arnaud Demes**  
**Day 1 · 13:30–15:00 · 20 minutes concepts + 70 minutes notebook**

In Lesson 04, six passages had already been prepared. Here we build the missing
layer: **raw financial document → ordered blocks → chunks → retrievable evidence**.

This notebook uses compact classroom fixtures based on official NVIDIA FY2026 and
Schneider Electric FY2025 disclosures. The fixtures keep the repository light and
the lab reproducible. Their official URLs and hashes remain attached throughout.

Runtime modes:

- `offline`: all parsers and chunkers run locally; LLM propositions are recorded;
- `ollama`: the proposition stage calls the configured local model;
- `openai`: the proposition stage calls the configured OpenAI model.

Parsing, fixed, recursive, structural, semantic and hierarchical chunking do not
need an LLM.

## Learning objectives

By the end of the laboratory, you can:

1. explain why PDF extraction is not yet document understanding;
2. normalize HTML and PDF into a canonical `DocumentBlock` model;
3. preserve pages, headings, tables, order and provenance;
4. compare fixed, recursive, structure-aware, semantic, hierarchical,
   contextual and LLM-assisted chunking;
5. reproduce a table-integrity failure caused by a bad boundary; and
6. select a strategy from measured evidence rather than intuition.

**Success condition:** the notebook compares seven strategies, reproduces the
naive failure and finishes with a verified provenance-preserving pipeline.

## Where this fits

```text
Lesson 03              Lesson 04                 Lesson 05
full context    →      prepared passages   →     real-source ingestion
                                                 ↓
Lesson 06       ←      retrievable chunks  ←     parsing + chunking
embeddings, hybrid retrieval and reranking
```

We keep the Lesson 04 lexical retriever constant. This isolates the effect of
changing chunk construction. The configured embedding provider creates semantic
boundaries here; hybrid ranking arrives in Lesson 06.

### Parser ladder

```text
raw bytes → text extraction → layout recovery → tables/headings → canonical blocks
 low semantic quality                                      high semantic quality
```

`pdfplumber` is a practical extraction baseline. It is not the final representation:
tables, coordinates, reading order, headings and provenance must still be verified.

### Laboratory method

```text
BUILD → OBSERVE → IMPROVE → VERIFY
parse    inspect    change policy    protect evidence
```


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from time import perf_counter
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pdfplumber
from matplotlib.patches import FancyBboxPatch

from finai_academy.chunking import (
    compare_chunking_strategies,
    contextual_enrich_chunks,
    contextualize_chunks,
    embedding_similarity_profile,
    fixed_chunks,
    hierarchical_chunks,
    proposition_chunks,
    recursive_chunks,
    semantic_chunks,
    structure_aware_chunks,
)
from finai_academy.documents import load_source_manifest, parse_html, parse_pdf
from finai_academy.hybrid_retrieval import DeterministicTeachingEmbeddings
from finai_academy.lesson_support import (
    RecordedChunkingModel,
    RecordedContextualChunkingModel,
)
from finai_academy.providers import create_chat_model, create_embeddings
from finai_academy.retrieval import EvidencePassage, LexicalRetriever
from finai_academy.settings import Settings

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "assets" / "course-data"
FIXTURES = DATA_ROOT / "fixtures"
LIVE_MODE = os.getenv("FINAI_LIVE_MODE", "0") == "1"

COLORS = {
    "navy": "#051C2A",
    "blue": "#1F40CB",
    "cyan": "#00A2EB",
    "orange": "#F07D00",
    "green": "#2E8B57",
    "grey": "#64748B",
    "light": "#E8EEF5",
}
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.titleweight": "bold"})
print("Runtime:", "live provider" if LIVE_MODE else "offline recorded mode")


### 1. Parser ladder and one canonical boundary

HTML exposes elements such as headings and tables. A PDF primarily exposes pages,
coordinates, characters and drawn lines. We therefore normalize both into the same
ordered record before discussing chunk size.

**Engineering rule:** provenance is created during parsing, not reconstructed after
retrieval. A more advanced parser (layout model, Docling or VLM) moves us up the
ladder only when measured extraction quality improves.


In [ ]:
stages = [
    ("Official source", "HTML or PDF"),
    ("Parser", "elements, pages, tables"),
    ("DocumentBlock", "order + structure + source"),
    ("Chunk strategy", "controlled boundaries"),
    ("Retriever", "evidence candidates"),
]
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(0, len(stages) * 2.2)
ax.set_ylim(0, 2.2)
ax.axis("off")
for index in range(len(stages) - 1):
    ax.annotate(
        "",
        xy=(index * 2.2 + 2.05, 1.1),
        xytext=(index * 2.2 + 1.75, 1.1),
        arrowprops={"arrowstyle": "->", "color": COLORS["cyan"], "lw": 2.5},
    )
for index, (title, subtitle) in enumerate(stages):
    x = index * 2.2 + 0.1
    box = FancyBboxPatch(
        (x, 0.55), 1.65, 1.1,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        facecolor=COLORS["navy"] if index == 2 else "white",
        edgecolor=COLORS["blue"], linewidth=2,
    )
    ax.add_patch(box)
    color = "white" if index == 2 else COLORS["navy"]
    ax.text(x + 0.825, 1.25, title, ha="center", va="center", weight="bold", color=color)
    ax.text(x + 0.825, 0.88, subtitle, ha="center", va="center", fontsize=8.5, color=color)
ax.set_title("Figure 1 — Parsing is a typed boundary, not a text-copy operation", pad=8)
plt.show()

In [ ]:
sources = load_source_manifest(DATA_ROOT / "manifest.json")
source_by_id = {source.source_id: source for source in sources}
for source in sources:
    print(f"{source.source_id}: fixture hash", "verified" if source.verify_fixture(REPO_ROOT) else "FAILED")

nvidia_blocks = parse_html(
    FIXTURES / "nvidia_fy2026_excerpt.html",
    source_by_id["NVDA-2026-10K-EXCERPT"],
)
schneider_blocks = parse_pdf(
    FIXTURES / "schneider_fy2025_excerpt.pdf",
    source_by_id["SU-2025-FY-EXCERPT"],
)
all_blocks = [*nvidia_blocks, *schneider_blocks]

block_frame = pd.DataFrame(
    {
        "block_id": block.block_id,
        "company": block.company,
        "ordinal": block.ordinal,
        "page": block.page_number,
        "type": block.block_type,
        "section": " > ".join(block.section_path),
        "characters": len(block.text),
        "preview": block.text[:72],
    }
    for block in all_blocks
)
block_frame

In [ ]:
type_colors = {
    "heading": COLORS["blue"],
    "paragraph": COLORS["cyan"],
    "table": COLORS["orange"],
    "list": COLORS["green"],
}
fig, axes = plt.subplots(2, 1, figsize=(13, 5.4), sharex=False)
for axis, (company, frame) in zip(axes, block_frame.groupby("company", sort=False), strict=True):
    for _, row in frame.iterrows():
        axis.scatter(
            row["ordinal"], 0,
            s=max(180, row["characters"] * 2.2),
            color=type_colors.get(row["type"], COLORS["grey"]),
            edgecolor="white", linewidth=1.5, zorder=3,
        )
        axis.text(row["ordinal"], 0, row["type"][0].upper(), ha="center", va="center", color="white", weight="bold")
        axis.text(row["ordinal"], -0.32, row["block_id"].split("-")[-1], ha="center", fontsize=8)
    axis.plot(frame["ordinal"], np.zeros(len(frame)), color=COLORS["light"], lw=5, zorder=1)
    axis.set_title(f"{company}: source order survives normalization", loc="left")
    axis.set_yticks([])
    axis.set_ylim(-0.6, 0.6)
    axis.spines[["left", "right", "top", "bottom"]].set_visible(False)
fig.suptitle("Figure 2 — Ordered blocks remain inspectable across HTML and PDF", y=1.01, weight="bold")
plt.tight_layout()
plt.show()

### 2. PDF text is linear; financial meaning is often two-dimensional

In a table, `EUR 40.2bn` is meaningful only when it remains linked to both the
`Revenue` row and the `FY2025` column. Reading order alone is insufficient.

`pdfplumber` gives us page text and detected table cells. The canonical block keeps
the normalized rows and the page number together.

In [ ]:
pdf_path = FIXTURES / "schneider_fy2025_excerpt.pdf"
with pdfplumber.open(pdf_path) as document:
    raw_page_text = document.pages[1].extract_text() or ""
normalized_table = next(block for block in schneider_blocks if block.block_type == "table")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
axes[0].axis("off")
axes[0].set_title("Linear extraction", loc="left")
axes[0].text(
    0, 0.98, raw_page_text,
    va="top", family="monospace", fontsize=9,
    bbox={"boxstyle": "round,pad=0.7", "facecolor": "#F8FAFC", "edgecolor": COLORS["light"]},
)
axes[1].axis("off")
axes[1].set_title("Canonical table block · page 2", loc="left")
table = axes[1].table(
    cellText=normalized_table.table_rows[1:],
    colLabels=normalized_table.table_rows[0],
    cellLoc="left", colLoc="left", loc="upper left", bbox=[0, 0.28, 1, 0.66],
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
for (row, _column), cell in table.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_facecolor("#DDEBFF" if row == 0 else "#F8FAFC")
    if row == 0:
        cell.set_text_props(weight="bold", color=COLORS["navy"])
axes[1].text(
    0, 0.13,
    f"source={normalized_table.source_id} · block={normalized_table.block_id} · page={normalized_table.page_number}",
    fontsize=9, color=COLORS["grey"],
)
fig.suptitle("Figure 3 — Preserve relationships before creating chunks", weight="bold")
plt.tight_layout()
plt.show()

### 3. Baselines before advanced strategies

We hold the parser output constant and vary only chunk construction.

- **Fixed:** transparent baseline; may cut anywhere.
- **Recursive:** prefers sentences and words.
- **Structure-aware:** treats tables atomically and keeps heading metadata.
- **Provider-aware semantic:** uses the configured embedding model to identify topic shifts.
- **Hierarchical:** retrieves small children and restores a larger parent.
- **Deterministic contextual:** prefixes company, period, document and section.
- **LLM contextual enrichment:** generates a short situating context while retaining raw text.

The seven core strategies all return `DocumentChunk`. Proposition chunking is an
optional transformation after the core laboratory has passed.


In [ ]:
construction_ms = {}

def measured(name, function):
    started = perf_counter()
    value = function()
    construction_ms[name] = (perf_counter() - started) * 1_000
    return value

def per_source(function, **kwargs):
    return [
        *function(nvidia_blocks, **kwargs),
        *function(schneider_blocks, **kwargs),
    ]

fixed = measured("fixed", lambda: per_source(fixed_chunks, chunk_size=90, overlap=10))
recursive = measured("recursive", lambda: per_source(recursive_chunks, max_chars=180))
structured = measured("structure", lambda: per_source(structure_aware_chunks, max_chars=260))
hierarchical = measured(
    "hierarchical",
    lambda: per_source(hierarchical_chunks, child_max_chars=140),
)
deterministic_context = measured("context_prefix", lambda: contextualize_chunks(structured))

if LIVE_MODE:
    settings = Settings.from_environment()
    embeddings = create_embeddings(settings)
    context_model = create_chat_model(settings)
    embedding_label = f"{settings.embedding_provider} / {settings.embedding_model}"
    context_label = f"{settings.provider} / {settings.chat_model}"
else:
    embeddings = DeterministicTeachingEmbeddings()
    recorded_contexts = {
        "NVDA-2026-10K-EXCERPT-STRUCTURE-001": "NVIDIA fiscal 2026 total revenue and annual growth disclosure.",
        "NVDA-2026-10K-EXCERPT-STRUCTURE-002": "NVIDIA fiscal 2026 revenue by business, including Data Center and Gaming.",
        "NVDA-2026-10K-EXCERPT-STRUCTURE-003": "NVIDIA fiscal 2026 revenue concentration interpretation and evidence limitation.",
        "SU-2025-FY-EXCERPT-STRUCTURE-001": "Schneider Electric FY2025 key financial metrics extracted from the results release.",
        "SU-2025-FY-EXCERPT-STRUCTURE-002": "Provenance note for the Schneider Electric FY2025 classroom extract.",
        "SU-2025-FY-EXCERPT-STRUCTURE-003": "Schneider Electric FY2025 revenue, organic growth and adjusted EBITA table.",
    }
    context_model = RecordedContextualChunkingModel(recorded_contexts)
    embedding_label = "offline / financial-concepts-v1"
    context_label = "offline / contextual-chunks-v1"

semantic_profiles = {}
def semantic_for(company, blocks):
    sentences, similarities = embedding_similarity_profile(blocks, embeddings)
    semantic_profiles[company] = (sentences, similarities)
    return semantic_chunks(blocks, threshold=0.20, similarities=similarities)

semantic = measured(
    "semantic",
    lambda: [
        *semantic_for("NVIDIA", nvidia_blocks),
        *semantic_for("Schneider Electric", schneider_blocks),
    ],
)

def enrich_all():
    nvidia_chunks = [chunk for chunk in structured if chunk.company == "NVIDIA"]
    schneider_chunks = [chunk for chunk in structured if chunk.company == "Schneider Electric"]
    return [
        *contextual_enrich_chunks(
            document_text="\n".join(block.text for block in nvidia_blocks),
            chunks=nvidia_chunks,
            model=context_model,
        ),
        *contextual_enrich_chunks(
            document_text="\n".join(block.text for block in schneider_blocks),
            chunks=schneider_chunks,
            model=context_model,
        ),
    ]

llm_contextual = measured("llm_contextual", enrich_all)

strategies = {
    "fixed": fixed,
    "recursive": recursive,
    "structure": structured,
    "semantic": semantic,
    "hierarchical": hierarchical,
    "context_prefix": deterministic_context,
    "llm_contextual": llm_contextual,
}
print("Embedding runtime:", embedding_label)
print("Context runtime:", context_label)
print("Seven strategies compared:", ", ".join(strategies))


In [ ]:
def nvidia_only(chunks):
    return [chunk for chunk in chunks if chunk.company == "NVIDIA"]

boundary_sets = {
    "Fixed · 90 chars": nvidia_only(fixed)[:7],
    "Recursive · 180 chars": nvidia_only(recursive)[:7],
    "Structure-aware · 260 chars": nvidia_only(structured)[:7],
}
fig, axes = plt.subplots(3, 1, figsize=(14, 7.5))
palette = [COLORS["blue"], COLORS["cyan"], COLORS["orange"], COLORS["green"]]
for axis, (label, chunks) in zip(axes, boundary_sets.items(), strict=True):
    cursor = 0
    total = sum(len(chunk.text) for chunk in chunks)
    for index, chunk in enumerate(chunks):
        width = len(chunk.text)
        axis.barh(0, width, left=cursor, height=0.55, color=palette[index % len(palette)], edgecolor="white")
        axis.text(cursor + width / 2, 0, f"{index + 1}\n{width}c", ha="center", va="center", color="white", fontsize=8, weight="bold")
        cursor += width
    axis.set_xlim(0, max(total, 1))
    axis.set_yticks([])
    axis.set_title(label, loc="left")
    axis.spines[["left", "right", "top"]].set_visible(False)
    axis.set_xlabel("characters shown in the first chunks")
fig.suptitle("Figure 4 — The same evidence receives different boundaries", weight="bold")
plt.tight_layout()
plt.show()

### 4. Provider-aware semantic boundaries

A semantic chunker compares adjacent sentence embeddings. A low similarity creates
a candidate boundary. The configured embedding provider is now part of the chunking
policy—not an implementation detail.

- high threshold → more boundaries and smaller chunks;
- low threshold → fewer boundaries and broader chunks.

Version the embedding provider, model, threshold and sentence segmentation because
all four can change the resulting chunks.


In [ ]:
sentences, similarities = semantic_profiles["NVIDIA"]
threshold = 0.20
boundaries = [index + 1 for index, score in enumerate(similarities) if score < threshold]

fig, ax = plt.subplots(figsize=(13, 4.6))
x = np.arange(1, len(similarities) + 1)
ax.plot(x, similarities, marker="o", lw=2.5, color=COLORS["blue"])
ax.axhline(threshold, color=COLORS["orange"], ls="--", lw=2, label=f"boundary threshold = {threshold:.2f}")
for boundary in boundaries:
    ax.axvspan(boundary - 0.12, boundary + 0.12, color=COLORS["orange"], alpha=0.18)
ax.set_xticks(x)
ax.set_xticklabels([f"S{i}→S{i+1}" for i in x])
ax.set_ylim(0, max(1.0, max(similarities, default=0) + 0.1))
ax.set_ylabel("adjacent embedding cosine similarity")
ax.set_title("Figure 5 — Provider-aware semantic boundaries", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print("Candidate boundaries after sentences:", boundaries)
print("Provider-aware semantic boundaries verified")


### 5. Hierarchical chunks separate retrieval granularity from reading context

Small child chunks are easier to match precisely. A parent section restores the
surrounding explanation after retrieval. The child must carry a stable `parent_id`;
otherwise expansion becomes guesswork.

In [ ]:
hierarchy = nvidia_only(hierarchical)
parent = next(chunk for chunk in hierarchy if chunk.role == "parent")
children = [chunk for chunk in hierarchy if chunk.parent_id == parent.chunk_id]

fig, ax = plt.subplots(figsize=(13, 5.2))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis("off")
child_x = np.linspace(1.2, 10.8, len(children))
for x in child_x:
    ax.plot([6, x], [4.25, 2.2], color=COLORS["cyan"], lw=2, zorder=1)
parent_box = FancyBboxPatch((3.7, 4.2), 4.6, 1.0, boxstyle="round,pad=0.04", facecolor=COLORS["navy"], edgecolor=COLORS["navy"], zorder=2)
ax.add_patch(parent_box)
ax.text(6, 4.7, f"PARENT · {len(parent.text)} chars\n{parent.section_path[-1]}", ha="center", va="center", color="white", weight="bold", zorder=3)
for index, (x, child) in enumerate(zip(child_x, children, strict=True), start=1):
    box = FancyBboxPatch((x - 0.8, 1.25), 1.6, 0.95, boxstyle="round,pad=0.03", facecolor="white", edgecolor=COLORS["blue"], linewidth=2, zorder=2)
    ax.add_patch(box)
    ax.text(x, 1.72, f"CHILD {index}\n{len(child.text)} chars", ha="center", va="center", fontsize=8.5, color=COLORS["navy"], zorder=3)
ax.text(6, 5.65, "Figure 6 — Retrieve children; expand the verified parent", ha="center", weight="bold", fontsize=13)
plt.show()

### 6. Deterministic prefix versus LLM contextual enrichment

A deterministic prefix copies trusted metadata. An LLM-generated context can add a
short explanation of where the passage sits inside the complete document.

**LLM contextual enrichment is not agentic chunking.** It is a bounded transformation:
one chunk in, one validated JSON context out. `generated_context` stays separate,
`raw_text` remains immutable evidence, and retrieval indexes both.


In [ ]:
example_raw = structured[0]
example_prefix = next(chunk for chunk in deterministic_context if chunk.chunk_id.endswith("001"))
example_enriched = next(chunk for chunk in llm_contextual if chunk.chunk_id == example_raw.chunk_id)

fig, axes = plt.subplots(1, 3, figsize=(15, 5.4))
cards = [
    ("Raw evidence", example_raw.text, COLORS["grey"]),
    ("Deterministic prefix", example_prefix.text, COLORS["cyan"]),
    ("LLM context + raw", example_enriched.text, COLORS["blue"]),
]
for axis, (title, text, color) in zip(axes, cards, strict=True):
    axis.axis("off")
    axis.set_title(title, loc="left", color=color)
    axis.text(
        0.02, 0.96, fill(text, 42), va="top", fontsize=9,
        bbox={"boxstyle": "round,pad=0.7", "facecolor": "white", "edgecolor": color, "linewidth": 2},
    )
fig.suptitle("Figure 7 — Generated context augments; it never replaces evidence", weight="bold")
plt.tight_layout()
plt.show()
print("Generated contextual enrichment verified")
print("Raw evidence preserved:", example_enriched.raw_text == example_raw.text)


### 7. Token inflation, construction latency and retrieval value

Generated context is useful only if its retrieval gain justifies extra index tokens,
prompt tokens, latency and transformation risk. We therefore expose both the cost and
the downstream retrieval result.

**Token inflation** below uses whitespace tokens as a transparent classroom proxy.
Production monitoring should use the selected model tokenizer.


In [ ]:
def whitespace_tokens(text):
    return len(text.split())

raw_tokens = sum(whitespace_tokens(chunk.text) for chunk in structured)
prefix_tokens = sum(whitespace_tokens(chunk.text) for chunk in deterministic_context)
llm_tokens = sum(whitespace_tokens(chunk.text) for chunk in llm_contextual)
token_counts = pd.Series(
    {"Raw structure": raw_tokens, "Metadata prefix": prefix_tokens, "LLM contextual": llm_tokens}
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0))
axes[0].bar(token_counts.index, token_counts.values, color=[COLORS["grey"], COLORS["cyan"], COLORS["blue"]])
axes[0].set_title("Index token proxy", loc="left")
axes[0].set_ylabel("whitespace tokens")
axes[0].tick_params(axis="x", rotation=15)
latency = pd.Series(construction_ms).sort_values()
axes[1].barh(latency.index, latency.values, color=COLORS["orange"])
axes[1].set_title("Observed construction latency", loc="left")
axes[1].set_xlabel("milliseconds")
axes[1].spines[["top", "right"]].set_visible(False)
fig.suptitle("Figure 8 — Enrichment has measurable construction and storage cost", weight="bold")
plt.tight_layout()
plt.show()
print(f"Token inflation measured: prefix={prefix_tokens/raw_tokens:.2f}x; llm={llm_tokens/raw_tokens:.2f}x")


### 8. Compare construction before comparing retrieval

A chunking scorecard should expose operational trade-offs before an answer is
generated. More chunks increase index size. Larger chunks consume more context.
Table integrity, raw-text preservation and provenance are hard requirements for
financial evidence.


In [ ]:
scorecard = compare_chunking_strategies(strategies, all_blocks)
scorecard["mean_chars"] = scorecard["mean_chars"].round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
axes[0].barh(scorecard["strategy"], scorecard["chunk_count"], color=COLORS["blue"])
axes[0].set_title("Chunk count", loc="left")
axes[0].set_xlabel("index records")
axes[0].invert_yaxis()
axes[0].spines[["top", "right"]].set_visible(False)
axes[1].barh(scorecard["strategy"], scorecard["mean_chars"], color=COLORS["cyan"])
axes[1].set_title("Mean chunk size", loc="left")
axes[1].set_xlabel("characters")
axes[1].invert_yaxis()
axes[1].spines[["top", "right"]].set_visible(False)
fig.suptitle("Figure 9 — Strategy changes storage and context economics", weight="bold")
plt.tight_layout()
plt.show()
scorecard

### 9. Retrieval comparison: hold the question and retriever constant

We reuse the Lesson 04 TF-IDF retriever. Each question has a literal evidence token
known to exist in the fixture. This is a small diagnostic set, not a production
benchmark.

The point is causal clarity: **same questions + same retriever + different chunks**.
Retrieval comparison is observational; provenance and evidence integrity remain the
provider-invariant release gates.


In [ ]:
questions = [
    ("Which NVIDIA business generated $193.7 billion?", "193.7"),
    ("How fast did NVIDIA Gaming revenue grow?", "41%"),
    ("What was Schneider Electric FY2025 revenue?", "40.2"),
    ("What margin did Schneider adjusted EBITA reach?", "18.7"),
]

def retrieval_recall(chunks):
    indexable = [chunk for chunk in chunks if chunk.role != "parent"]
    passages = [
        EvidencePassage(
            passage_id=chunk.chunk_id,
            company=chunk.company,
            period=chunk.period,
            section=" > ".join(chunk.section_path) or "Document",
            text=chunk.text,
            source_url=chunk.source_url,
        )
        for chunk in indexable
    ]
    retriever = LexicalRetriever(passages)
    recovered = 0
    for question, token in questions:
        hits = retriever.search(question, top_k=min(2, len(passages)))
        recovered += any(token in hit.passage.text for hit in hits)
    return recovered / len(questions)

scorecard["retrieval_recall_at_2"] = [
    retrieval_recall(strategies[strategy]) for strategy in scorecard["strategy"]
]

heat_columns = [
    "heading_retention",
    "table_integrity",
    "provenance_completeness",
    "retrieval_recall_at_2",
]
matrix = scorecard.set_index("strategy")[heat_columns]
fig, ax = plt.subplots(figsize=(12.5, 6.2))
image = ax.imshow(matrix.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(heat_columns)))
ax.set_xticklabels(["headings", "tables", "provenance", "retrieval@2"])
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        value = matrix.iloc[row, column]
        ax.text(column, row, f"{value:.0%}", ha="center", va="center", color="white" if value > 0.55 else COLORS["navy"], weight="bold")
ax.set_title("Figure 10 — One strategy rarely dominates every engineering metric", loc="left")
fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03, label="pass rate")
plt.tight_layout()
plt.show()
scorecard
print("Retrieval comparison complete")

## Failure lab

A fixed character window can separate a table's header from its values. Retrieval
may still find a number, but the model cannot safely know which row or column gives
that number meaning.

Diagnose the stage before changing the model:

1. Did `pdfplumber` recover the row and column correctly?
2. Did chunk construction keep them together?
3. Only then: did retrieval rank the coherent chunk?

In [ ]:
table_block = next(block for block in nvidia_blocks if block.block_type == "table")
fixed_table = [chunk for chunk in fixed if table_block.block_id in chunk.source_block_ids]
structured_table = next(chunk for chunk in structured if table_block.block_id in chunk.source_block_ids)
table_is_split = not any(table_block.text in chunk.text for chunk in fixed_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].axis("off")
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].set_title("Fixed windows · relationship broken", loc="left", color=COLORS["orange"])
y = 0.82
for index, chunk in enumerate(fixed_table, start=1):
    display_text = fill(chunk.text.replace(chr(36), "USD "), width=48)
    axes[0].text(
        0.02, y, f"F{index}\n{display_text}", va="top", family="monospace", fontsize=10,
        bbox={"boxstyle": "round,pad=0.55", "facecolor": "#FFF4E8", "edgecolor": COLORS["orange"], "linewidth": 1.8},
    )
    y -= 0.38
axes[0].annotate(
    "Boundary separates the final Gaming cells",
    xy=(0.35, 0.40), xytext=(0.15, 0.08),
    arrowprops={"arrowstyle": "->", "color": COLORS["orange"], "lw": 2},
    color=COLORS["orange"], weight="bold",
)
axes[1].axis("off")
axes[1].set_title("Structure-aware · table atomic", loc="left", color=COLORS["green"])
display_rows = [
    [cell.replace(chr(36), "USD ") for cell in row]
    for row in table_block.table_rows
]
intact_table = axes[1].table(
    cellText=display_rows[1:], colLabels=display_rows[0],
    cellLoc="left", colLoc="left", bbox=[0.0, 0.34, 1.0, 0.52],
)
intact_table.auto_set_font_size(False)
intact_table.set_fontsize(9.5)
for (row, _column), cell in intact_table.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_facecolor("#DDF3E5" if row == 0 else "#F2FAF5")
    if row == 0:
        cell.set_text_props(weight="bold", color=COLORS["navy"])
axes[1].text(0, 0.21, "One chunk · complete row/column relationship", fontsize=10, color=COLORS["green"], weight="bold")
axes[1].text(0, 0.11, f"source block: {structured_table.source_block_ids[0]}", fontsize=9, color=COLORS["grey"])
fig.suptitle("Figure 11 — A retrieval-ready number can still be analytically unsafe", weight="bold")
plt.tight_layout()
plt.show()

assert table_is_split
assert table_block.text in structured_table.text
print("Table integrity failure reproduced")
print("Diagnosis: chunk construction failed before retrieval or generation")

## Verification

We verify the non-negotiable properties before any optional transformation:

- all seven core strategy outputs retain source identifiers and URLs;
- structure-aware chunks keep parsed tables atomic;
- provider-aware semantic boundaries return one score per adjacent sentence;
- LLM contextual chunks preserve raw evidence and store generated context separately;
- token inflation is visible rather than hidden; and
- the fixed baseline demonstrably loses table integrity.

Retrieval recall remains a comparative metric, not a universal pass/fail threshold.


In [ ]:
provenance_ok = all(
    chunk.source_block_ids and chunk.source_url
    for chunks in strategies.values()
    for chunk in chunks
)
structure_row = scorecard.loc[scorecard["strategy"] == "structure"].iloc[0]
fixed_row = scorecard.loc[scorecard["strategy"] == "fixed"].iloc[0]
raw_text_ok = all(
    enriched.raw_text == original.text
    for enriched, original in zip(llm_contextual, structured, strict=True)
)
generated_context_ok = all(chunk.generated_context for chunk in llm_contextual)
semantic_shape_ok = all(
    len(similarities) == max(0, len(sentences) - 1)
    for sentences, similarities in semantic_profiles.values()
)

checks = {
    "seven strategies": len(strategies) == 7,
    "provenance complete": provenance_ok,
    "structure keeps tables": structure_row["table_integrity"] == 1.0,
    "fixed failure visible": fixed_row["table_integrity"] < 1.0,
    "semantic profile complete": semantic_shape_ok,
    "generated context separate": generated_context_ok,
    "raw evidence preserved": raw_text_ok,
    "token inflation visible": llm_tokens > raw_tokens,
    "fixture hashes verified": all(source.verify_fixture(REPO_ROOT) for source in sources),
}
for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {label}")
assert all(checks.values())
print("Provider-aware semantic boundaries verified")
print("Generated contextual enrichment verified")
print("Raw evidence preserved")
print("Token inflation measured")
print("Retrieval comparison complete")
print("Table integrity failure reproduced")
print("Seven strategies compared")
print("PASS — document and chunking laboratory verified")


### Optional extension: proposition chunking

After the core PASS marker, compare LLM-generated atomic propositions. This is a
lossy transformation and is not a default evidence store: validate every numeric
qualifier, preserve source block IDs and retain the original text.

Docling/VLM parsing and iterative agentic grouping are also optional extensions;
they do not affect the classroom PASS contract.


In [ ]:
RUN_OPTIONAL_PROPOSITIONS = os.getenv("FINAI_RUN_OPTIONAL", "0") == "1"
if RUN_OPTIONAL_PROPOSITIONS:
    proposition_model = create_chat_model(Settings.from_environment()) if LIVE_MODE else RecordedChunkingModel()
    proposition_input = [block for block in all_blocks if block.block_type == "paragraph"]
    propositions = proposition_chunks(proposition_input, proposition_model)
    print("Optional propositions generated:", len(propositions))
else:
    print("Optional proposition chunking skipped; set FINAI_RUN_OPTIONAL=1 to run it.")


## Challenge

Select one initial production policy for the Financial Analyst Copilot and defend it
in four lines:

1. Which chunk representation will be indexed?
2. Which parent or neighboring context will be restored after retrieval?
3. Which metadata must be filtered before ranking?
4. Which maintained questions and integrity checks must pass before release?

**Suggested answer:** index structure-aware child chunks with company, period,
document type, section, page and source metadata; preserve a parent section ID for
expansion; keep tables atomic; use contextual prefixes only when they improve the
maintained retrieval set enough to justify extra tokens. Treat proposition chunks as
an evaluated secondary representation, not a replacement for original evidence.

## Capstone integration

The Financial Analyst Copilot now gains a configurable ingestion boundary:

```text
source manifest
  → HTML/PDF parser
  → canonical DocumentBlock records
  → selected chunking policy
  → provenance-preserving DocumentChunk records
```

Reusable implementation:

- `src/finai_academy/documents.py`
- `src/finai_academy/chunking.py`
- `assets/course-data/manifest.json`

The application can now change its chunking strategy without rewriting parsing or
retrieval code.

The selected representation now has an explicit provider/model/threshold identity,
separate raw and generated fields, and measured construction/retrieval trade-offs.


## Recap

- `pdfplumber` is an extraction baseline; parsing quality must be visually verified.
- Canonical blocks preserve order, structure and provenance across formats.
- Fixed and recursive chunking are useful baselines, not production defaults.
- Structure-aware chunks protect headings and tables.
- Semantic boundaries depend on the configured embedding model and threshold.
- Hierarchical chunks separate retrieval precision from reading context.
- Deterministic prefixes and LLM contextual enrichment are different policies.
- Generated context augments `raw_text`; it never replaces source evidence.
- Token, latency and retrieval gains must be measured together.

### Knowledge check

1. Why is contextual enrichment not agentic chunking?  
   It is one bounded, validated transformation without autonomous planning or tool loops.
2. What must remain immutable after LLM enrichment?  
   Raw evidence, stable chunk ID, page/block provenance and source URL.
3. When is semantic chunking reproducible?  
   When segmentation, embedding provider/model, threshold and source version are recorded.
